In [ ]:
# === STANDART COLAB KURULUM HÜCRESİ ===
from google.colab import drive, userdata
drive.mount("/content/drive")

import os, sys
REPO_URL = "https://github.com/Oldzhay34/receipts-and-interactions.git"
BRANCH   = "main"
try:
    TOKEN = userdata.get("GH_TOKEN")
except Exception:
    TOKEN = None
url = REPO_URL.replace("https://", f"https://{TOKEN}@") if TOKEN else REPO_URL

if not os.path.exists("/content/repo"):
    !git clone -q -b {BRANCH} {url} /content/repo
else:
    !cd /content/repo && git fetch -q && git checkout -q {BRANCH} && git pull -q

%cd /content/repo
!pip install -q -r requirements.txt && pip install -q -e .

if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")

os.environ["DATA_ROOT"] = "/content/drive/MyDrive/CME4403/data"
from recsys.config import load_config
CFG = load_config()
print("commit:", os.popen("git rev-parse --short HEAD").read().strip())

# 06 — Visualizations and distribution analysis

Stage 1.6. All figures use `recsys.viz.style`; plot functions return a Figure and never save.
Everything is saved once, as one `figures` version (file names `fig_06_NN_...`).
The finding drafts under each block cite numbers from the outputs; values in `{braces}` must be
filled in from the cell output above them.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

from recsys.io.artifact_store import ArtifactStore
from recsys.analysis.distribution import (
    cold_start_summary,
    distribution_summary_table,
    list_items,
    present_columns,
    rating_by_list_item,
    reviews_per_entity,
    with_rating_stats,
)
from recsys.contracts import (
    FIGURES_STAGE_SCHEMAS,
    LOADED_INTERACTIONS_SCHEMA,
    LOADED_RECIPES_SCHEMA,
    validate,
)
from recsys.viz.plots import (
    bar_counts,
    box_by_group,
    corr_heatmap,
    hist_box,
    long_tail_hist,
    scatter_sample,
)

store = ArtifactStore.from_config(CFG)

# Input stage: `loaded`. The `missing` (1.3) and `outliers` (1.4) stages store evidence tables,
# not cleaned recipe / interaction tables (their transformers are fitted on train data in the
# 1.9 pipeline), so the figures are drawn from `loaded` and the manifest records exactly that.
loaded = store.load("loaded")
upstream = loaded
input_stage_used = "loaded"
recipes = validate(loaded["recipes"], LOADED_RECIPES_SCHEMA)
interactions = validate(loaded["interactions"], LOADED_INTERACTIONS_SCHEMA)

print(f"Using input stage: {input_stage_used} ({upstream.version})")
print(f"Recipes: {recipes.shape}, Interactions: {interactions.shape}")

In [ ]:
numeric_cols = present_columns(recipes)  # minutes, n_steps, n_ingredients, 7 nutrition

dist_table = distribution_summary_table(recipes, numeric_cols)
display(dist_table)

figures = {}
fig_idx = 1

for col in numeric_cols:
    key_raw = f"fig_06_{fig_idx:02d}_{col}_raw"
    figures[key_raw] = hist_box(recipes[col], title=f"{col} (Raw)", log1p=False)
    fig_idx += 1

    key_log = f"fig_06_{fig_idx:02d}_{col}_log1p"
    figures[key_log] = hist_box(recipes[col], title=f"{col} (log1p)", log1p=True)
    fig_idx += 1

print(f"Generated {len(figures)} numeric distribution figures.")



**Findings (draft) — numeric distributions (fig_06_01 … fig_06_20).**
Every numeric feature except `n_ingredients` is classified as long-tail (`distribution_summary`:
9 long-tail, 1 right-skewed), so the raw scale is dominated by a few extreme recipes; on the
log1p scale they are far less skewed, which is why 1.4 / 1.8 apply log1p before scaling.
`minutes` has the strongest skew (skewness {distribution_summary: minutes skewness}).


In [ ]:
# 1. Rating counts (class imbalance, including 0 = unrated)
figures[f"fig_06_{fig_idx:02d}_rating_counts"] = bar_counts(
    interactions["rating"],
    top_n=6,
    title="Interaction Rating Counts (0 = Unrated)",
    xlabel="Count",
    ylabel="Rating",
    horizontal=False,
)
fig_idx += 1

# 2. Top 20 ingredients & tags
figures[f"fig_06_{fig_idx:02d}_top20_ingredients"] = bar_counts(
    list_items(recipes["ingredients"], "ingredient"), top_n=20, title="Top 20 Raw Ingredients"
)
fig_idx += 1

figures[f"fig_06_{fig_idx:02d}_top20_tags"] = bar_counts(
    list_items(recipes["tags"], "tag"), top_n=20, title="Top 20 Recipe Tags"
)
fig_idx += 1

# 3. Long-tail interaction counts & cold-start statistics
user_counts, recipe_counts = reviews_per_entity(interactions)
cold_start = cold_start_summary(user_counts, recipe_counts)
display(cold_start)
print(
    "Cold-start finding:",
    cold_start.set_index("entity").loc["user", "single_review_pct"],
    "% of users have only 1 review.",
)

figures[f"fig_06_{fig_idx:02d}_user_reviews_long_tail"] = long_tail_hist(
    user_counts, log=True, title="Reviews per User (Long-Tail)", xlabel="Reviews per User"
)
fig_idx += 1

figures[f"fig_06_{fig_idx:02d}_recipe_reviews_long_tail"] = long_tail_hist(
    recipe_counts, log=True, title="Reviews per Recipe (Long-Tail)", xlabel="Reviews per Recipe"
)
fig_idx += 1


**Findings (draft) — ratings, top items and long tail (fig_06_21 … fig_06_25).**
Ratings are highly imbalanced: 5 stars make up 76.2 % of rated interactions, and 0 ("no stars
given", 60,847 rows) is treated as missing, not as the lowest score — so accuracy alone is not
a useful metric. The most frequent ingredients are pantry staples ({top ingredient 1}, {top
ingredient 2}, {top ingredient 3}), which carry little signal for recommendation.
**Cold start:** 73.38 % of users (166,256 of 226,570) and 39.70 % of recipes (91,953 of
231,637) have exactly one review (`cold_start_summary`), so collaborative signals are missing
for most users and the content-based kNN (ingredients) has to carry the recommendation.


In [ ]:
# Exploratory avg_rating (excluding rating == 0; model targets are computed on train-only in 1.9)
recipes_viz = with_rating_stats(recipes, interactions)

# Scatter plots
figures[f"fig_06_{fig_idx:02d}_scatter_ingredients_minutes"] = scatter_sample(
    recipes_viz, "n_ingredients", "minutes", n=20_000, alpha=0.2, log_y=True,
    title="Preparation Minutes vs Number of Ingredients (log y)"
)
fig_idx += 1

figures[f"fig_06_{fig_idx:02d}_scatter_calories_rating"] = scatter_sample(
    recipes_viz, "calories", "avg_rating", n=20_000, alpha=0.2, log_x=True,
    title="Average Rating vs Calories (log x)"
)
fig_idx += 1

figures[f"fig_06_{fig_idx:02d}_scatter_steps_rating"] = scatter_sample(
    recipes_viz, "n_steps", "avg_rating", n=20_000, alpha=0.2,
    title="Average Rating vs Number of Steps"
)
fig_idx += 1

# Correlation heatmap (Spearman for skewed features)
heatmap_cols = numeric_cols + ["avg_rating", "n_ratings"]
figures[f"fig_06_{fig_idx:02d}_corr_heatmap"] = corr_heatmap(
    recipes_viz, cols=heatmap_cols, method="spearman", title="Spearman Correlation Heatmap"
)
fig_idx += 1

# Box plot of avg_rating by top 10 culinary tags
tag_rating_df = rating_by_list_item(recipes_viz, "tags", "avg_rating")
figures[f"fig_06_{fig_idx:02d}_box_rating_by_tag"] = box_by_group(
    tag_rating_df, value="avg_rating", group="tags", top_n_groups=10,
    title="Average Rating by Top 10 Recipe Tags"
)
fig_idx += 1

print(f"Total figures ready to save: {len(figures)}")



**Findings (draft) — relationships (fig_06_26 … fig_06_30).**
Preparation time grows only weakly with the number of ingredients (Spearman
{corr minutes–n_ingredients}); average rating is nearly flat across calories
({corr calories–avg_rating}) and steps ({corr n_steps–avg_rating}), so recipe size and health
features alone do not predict ratings. Nutrition columns are strongly correlated with each other
(e.g. calories–total_fat {corr}), which matters for distance weighting in 1.8 / 1.10. Average
rating differs little between the top tags (medians {box medians}).


In [ ]:
inputs_list = [upstream] if upstream.stage == "loaded" else [upstream, loaded]

objects_to_save = {
    "distribution_summary": dist_table,
    "cold_start_summary": cold_start,
    **figures,
}

saved = store.save(
    stage="figures",
    objects=objects_to_save,
    inputs=inputs_list,
    schemas=FIGURES_STAGE_SCHEMAS,  # tables validated before writing
    params={
        "input_stage_used": input_stage_used,
        "input_version": upstream.version,
        "n_figures": len(figures),
        "scatter_sample_n": 20_000,
        "corr_method": "spearman",
        "dpi": 150,
    },
)

for fig in figures.values():
    plt.close(fig)

print("Saved figures artifact:", saved.path)
print("Version:", saved.version, "| Input stage:", input_stage_used)